# 第 08 课 · 那个"乘二"是从哪来的（浏览器版）

配套课文：[第 08 课 · 乘二从哪来（网页版）](../docs/m0-foundations/08-where-the-times-two-comes-from.md)。

**怎么用**：本地装了 Jupyter 就直接打开这个文件；不想装任何东西，打开 [https://jupyter.org/try](https://jupyter.org/try)（或 Colab），把这一页整份放进去，每一格按 `Shift + Enter` 从上往下跑。

只用 Python 标准库，不需要安装任何东西。

**每格跑出来的数字和课文里印的逐字一致**（种子钉在 7）。

## 一、第 07 课让你乘了二，但没跟你讲为什么

第 07 课给你的那台计算器，最后一行是：

```
真实水平大概落在 32 到 39 之间
```

那个"大概"里面藏着一次乘法：**毛边乘二**。我当时说"为什么乘二而不是乘三，是下一课的内容，今天你先当成一条能用的规矩"。

你现在有权利要一个解释了。而且这个解释很重要，因为它决定你能不能把这句话改成别的说法——比如"我有九成的把握"，或者"我可以说这个涨幅是真的吗"。乘二只是最外面那一层皮，里面是一件你可能从没亲眼见过的事。

所以这一节课不推导，我们**演**。这是这门课第二次用蒙特卡洛，你会发现它的形状和第一次一模一样：不清楚一件事到底什么样，就造很多个平行世界，让每件事在里面发生一遍，然后去数。

## 二、把"平均"这件事演两万遍，看它长什么形状

回忆第 07 课最核心那句话：你算出来的平均不是一个固定的数，而是一次抽签的结果。这句话你可能还有点不服——它凭什么？

那就把这件事演出来。用你手边那列十天通勤记录：`34、41、29、38、45、31、37、40、26、33`。它的平均是 35.4。

现在做一件现实中永远不会发生的事：**假装有无数个"十天"**，从这十个数里每次抽十条（抽完放回，所以下一次还是这十个数的脾气），算一个平均，记录下来。这样得到两万个平均。

先把三台机器和两列记录造好（下面这一格只是定义，运行它不会打印任何东西）。`repeat_averages` 就是"抽 k 条取平均，重复 runs 遍"，`spread_of` 量一列数的起伏，`histogram` 把一列数按大小分堆、排成字符图：

In [1]:
import math
import random


def spread_of(record):
    n = len(record)
    m = sum(record) / n
    return math.sqrt(sum((x - m) ** 2 for x in record) / (n - 1))


def repeat_averages(data, k, runs):
    out = []
    for _ in range(runs):
        sample = random.choices(data, k=k)
        out.append(sum(sample) / k)
    return out


def histogram(values, bins=9, width=46):
    lo = min(values)
    hi = max(values)
    step = (hi - lo) / bins
    counts = [0] * bins
    for x in values:
        k = int((x - lo) / step)
        if k == bins:
            k = bins - 1
        counts[k] = counts[k] + 1

    biggest = max(counts)
    for i in range(bins):
        left = lo + i * step
        bar = "*" * max(1, int(round(counts[i] / biggest * width)))
        print(f"{left:6.1f}  {bar}  {counts[i]}")


taxi = [3, 5, 2, 4, 30, 3, 2, 5]
commute = [34, 41, 29, 38, 45, 31, 37, 40, 26, 33]

代码在 [code/lesson08_error_bars.py](../code/lesson08_error_bars.py)。核心是那台你已经很熟的机器，只换了要数的东西。`random.choices(commute, k=10)` 是"从这列数里带放回抽十条"，`avgs` 收集两万条平均：

In [2]:
random.seed(7)
avgs = []
for _ in range(20000):
    sample = random.choices(commute, k=10)
    avgs.append(sum(sample) / 10)

然后把这两万条数按大小分堆、画成图——图是这段代码自己排的字符，不需要任何别的工具。

我这台机器跑出来的形状（横着看，每行是一个时长区间，`*` 的长短是落在里面的次数）：

```
  29.0  *  43
  30.4  ***  418
  31.8  **************  1854
  33.2  **********************************  4506
  34.6  **********************************************  6077
  36.1  ***********************************  4632
  37.5  ***************  1989
  38.9  ***  434
  40.3  *  47
```

In [3]:
histogram(avgs)

  29.0  *  43
  30.4  ***  418
  31.8  **************  1854
  33.2  **********************************  4506
  34.6  **********************************************  6077
  36.1  ***********************************  4632
  37.5  ***************  1989
  38.9  ***  434
  40.3  *  47


**中间鼓，两头瘦，左右大致对称。** 这个形状有个名字叫**钟形**（也叫正态）。

注意你在上面这张图里看到的东西：**两万个"平均"自己排成了一个漂亮的形状，而你手上的原始记录完全不是这个形状。** 原始那十天是散在 26 到 45 之间的一把数；平均却老老实实堆在 35 附近，越远越少。

同一件事画成图如下。蓝色柱子是两万条平均的分堆，底部那一排橙色短竖线是原始那十条钉在横轴上的位置——同一把尺子（分钟），两种形状。

![两万个"平均"排成钟形：蓝色柱子是两万条"抽十条取平均"的分堆，底部橙色短竖线是原始那十条记录；黑实线是平均 35.4，两条虚线是 ±1 倍和 ±2 倍毛边，分别盖住 70.2% 和 96.6%](https://raw.githubusercontent.com/JavenZheng/monte-carlo-finance/main/assets/fig08_central_limit.svg)

图由 [code/fig08_central_limit.py](../code/fig08_central_limit.py) 生成，和上面那段字符图用的是同一轮抽签（种子 7）。

这就是第 07 课那句"平均是一次抽签"的真面目：它不是随便乱飘的抽签，它有形状，而且形状是可以演出来给你看的。

## 三、钟形这份礼物：它把"乘几"变成了"几成把握"

形状一旦知道，倍数就不再是拍脑袋的规矩，而是有对价的东西。我拿上面那两万条平均直接去数：

In [4]:
se = 1.856
for mult in (1, 2, 3):
    inside = 0
    for a in avgs:
        if abs(a - 35.4) <= mult * se:
            inside = inside + 1
    print(f"±{mult} 倍毛边（±{mult * se:.1f} 分钟）之内：{inside / len(avgs) * 100:.1f}%")

±1 倍毛边（±1.9 分钟）之内：70.2%
±2 倍毛边（±3.7 分钟）之内：96.6%
±3 倍毛边（±5.6 分钟）之内：99.9%


**这就是那三个数的来历。** 毛边乘一，覆盖七成；乘二，覆盖九成五以上；乘三，几乎全覆盖。以后你在任何场合看到"误差棒 ±2"之类的写法，它的意思就是这一行数：我给你的这个范围，用同样的方法重复做一百次，大概有九十五次会把真值罩在里面。

所以"乘二"不是魔法数字，是你用**四分的把握换七成的把握**（从 70% 到 96.6%）买到的那个价格。你以后完全可以选别的倍数，只是要报出你选它换来的把握——**这是这一节课真正想给你的东西：一个可以谈判的旋钮，而不是一条规矩。**

顺便验证一下第 07 课那个式子。它说毛边 = 起伏 ÷ √条数。我这台机器算出来：

In [5]:
print(f"预测的毛边 {spread_of(commute) / math.sqrt(10):.2f}，"
      f"实测两万条平均的散布 {spread_of(avgs):.2f}")

预测的毛边 1.86，实测两万条平均的散布 1.77


式子给的 1.86，和把两万个平均排出来量它们的散布 1.77，差了五个百分点左右。这两个数本来就该接近——式子说的是"这件事的脾气"，模拟是"真的做一遍"。**它们是同一件事的两种看法，而且它们对上了。**

（那五个百分点的差也不是错。量"散布"本身也要靠抽签，两万次抽出来的散布自己也有毛边。这条以后你会在方差缩减那一章见到正式版。）

## 四、为什么平均会变钟形，而你手上的数不会

这件事有个名字，叫**中心极限定理**。它的直觉版本只有一句话，而且你其实已经在第 07 课的进阶框里见过它：

**很多个互不相干的小波动加起来，形状会趋向钟形——不管每个小波动自己是什么形状。**

你的平均正是这样一次加总：十条记录相加再除以十。每一条各自抽自己的，互不影响（这就是"独立"那两个字的用途，也是第 01 课那句"抽完必须放回"背后在保护的东西）。加总之后的形状，比任何单个成员都要规矩得多。

原始记录为什么不钟形？因为它不是加总，它是**单次抽签**。你某一次等车 30 分钟，那一次不需要对任何人负责。钟形是"很多次一起"才出现的规律，单个成员享受不到它。

这一点对后面很重要。原书有一整套技术用来对付"这个平均到底有多准"，它们全都建立在钟形之上——而钟形的代价是：**你必须真的在算某种加总。** 你要估计的东西如果是"最坏那一次"，它就不在钟形管辖之内（第三节末尾那个"两端的数飘"就是前兆）。

## 五、它什么时候不灵

我把同一套动作换到那列很歪的等车记录上（`3、5、2、4、30、3、2、5`，八次数里有一次等了半小时），同样抽十条取平均，重复一万遍。跑出来的形状是：

```
   2.3  **********************************************  2553
   4.2  *****************************************  2250
   6.1  *****************************  1620
   8.0  *******************************************  2370
   9.9  ***************  834
  11.9  **  112
  13.8  ****  216
  15.7  *  41
  17.6  *  4
```

In [6]:
random.seed(7)
avgs2 = repeat_averages(taxi, 10, 10000)
histogram(avgs2)
print(f"预测的毛边 {spread_of(taxi) / math.sqrt(10):.2f}，"
      f"实测一万条平均的散布 {spread_of(avgs2):.2f}")

   2.3  **********************************************  2553
   4.2  *****************************************  2250
   6.1  *****************************  1620
   8.0  *******************************************  2370
   9.9  ***************  834
  11.9  **  112
  13.8  ****  216
  15.7  *  41
  17.6  *  4
预测的毛边 2.99，实测一万条平均的散布 2.81


**这不是钟。这是两个包。** 左边那一大坨是"这次十条里没抽到那个 30"（大多数时候都在 2.3 到 6 之间），右边那鼓包是"这次抽到了 30"（平均一下被抬到 8 以上）。中间那段 6.1 反而是低谷。

钟形呢？它还没来。原因不是定理错了，是十条太少：一个平均要"平均掉"很多小波动，才会规矩地堆在中间；当某一次抽签里的一个成员能顶起整个结果（30 相对其他七个数就是这种成员），十条根本不够。我把它改成抽一百条再跑，形状就变成中间鼓两头瘦了，把"乘二"的覆盖数也变回九成六：

```
   3.8  *  21
   4.5  ******  381
   5.3  ************************  1632
   6.0  **********************************************  3125
   6.7  *****************************************  2797
   7.5  **********************  1477
   8.2  *******  465
   8.9  *  92
   9.7  *  10
```

In [7]:
random.seed(7)
avgs3 = repeat_averages(taxi, 100, 10000)
histogram(avgs3)
se100 = spread_of(taxi) / math.sqrt(100)
truth = sum(taxi) / len(taxi)
inside = 0
for a in avgs3:
    if abs(a - truth) <= 2 * se100:
        inside = inside + 1
print(f"±2 倍毛边（±{2 * se100:.1f} 分钟）之内：{inside / len(avgs3) * 100:.1f}%")

   3.8  *  21
   4.5  ******  381
   5.3  ************************  1632
   6.0  **********************************************  3125
   6.7  *****************************************  2797
   7.5  **********************  1477
   8.2  *******  465
   8.9  *  92
   9.7  *  10
±2 倍毛边（±1.9 分钟）之内：96.6%


**这就是你要带走的第二个结论，而且它在生活里比第一个更有用：条数少 + 数据歪，这两件事叠在一起的时候，第 07 课那个"起伏 ÷ √条数"和你那个"乘二"会一起失真。** 而你的真实数据，绝大多数都是歪的：播放量、发票金额、订单大小、支出、等待时长——它们全都长得像那列等车记录，不像通勤记录。

这不是这门课的边角问题。原书整整一节（§9.3）处理"厚尾"这种形状，因为金融里最贵的错误几乎都是它带来的：**你以为你有一百条数据，所以你的估计很稳；其实你有一条数据，其他九十九条只是陪衬。**

## 六、还有一件事，它管不了

现在把两节课拼起来。

你算出"我的平均通勤 35.4 分钟，毛边 1.9，乘二之后范围 31.7 到 39.1"。这句话的正确读法只有一个：**如果我的记录方式是诚实的，那么真值大概率在这个范围里。**

它没有覆盖的是"我的记录方式本身歪了"。你只记工作日的通勤，那 35.4 是"工作日的水平"，无论你抽两万遍、把范围缩到多窄，它都不会靠近"我每天出门花多久"。第 07 课第八节说的那个瞄准镜，我们当时只交代了它存在；到这里可以说得更准一点：

**毛边、乘二、置信区间这一整套，量的是"我抽到的这两万个平均有多分散"。而你抽样的对象本身是不是你想问的那个东西，这套东西一个字都不管。**

原书第 7 章第一节（§7.1.1）用"偏差与方差"两个词正面处理这件事，因为估计一个希腊字母时，这两种错会同时存在，而且常常互相掩盖：**你把方差压得很小，看起来精确到小数点后四位，其实那个数是歪的。** 你以后会看到很多这样的表。

## 七、今晚就能用的：一句带把握的话

那台计算器就在文件的第一部分，给你的任何一列记录加一根误差棒（下面这一格是它的定义，运行它不会打印任何东西）：

In [8]:
def fuzz(records, unit=""):
    n = len(records)
    mean = sum(records) / n
    spread = math.sqrt(sum((x - mean) ** 2 for x in records) / (n - 1))
    se = spread / math.sqrt(n)
    print(f"平均 {mean:.1f}{unit} ± {se:.1f}（毛边）")
    print(f"你敢报的范围：{mean - 2 * se:.1f} 到 {mean + 2 * se:.1f}{unit}")

它跑那列等车记录（`taxi`，第五节那八条）输出：

In [9]:
fuzz(taxi, " 分钟")

平均 6.8 分钟 ± 3.3（毛边）
你敢报的范围：0.1 到 13.4 分钟


看这一行，你就知道这一课不是白上的：**范围宽到 0.1 到 13.4，八条数据里有 30 那种东西，这套办法给你的诚实答案就是"我什么也说不了"。** 换成通勤那十天，范围是 31.7 到 39.1，你才敢说一句"我大概三十五分钟，可能差三四分钟"。

In [10]:
fuzz(commute, " 分钟")

平均 35.4 分钟 ± 1.9（毛边）
你敢报的范围：31.7 到 39.1 分钟


于是你在会上最有用的那句话，从"我们测出来涨了 500"变成：

**"我们测出来涨了 500，毛边 670，所以这个涨幅还没出噪声范围。要分辨 500 的差距，每组要一百条。"**

三件套：**一个数、它的毛边、要多少条才能看清你在意的那个差距。** 这三样齐全的话，没人能用"你这个数怎么这么好看"把你带走。

## 这一课带走的一句话

乘二不是规矩，是用四成把握换九成五把握的那个价格；而它只在"很多次加在一起"的那个数上成立——你的数据越歪，要加的次数越多。

配套课文：[第 08 课 · 乘二从哪来（网页版）](../docs/m0-foundations/08-where-the-times-two-comes-from.md)。

---

**许可**：正文 CC BY-NC-SA 4.0，代码 MIT，见 [LICENSE.md](../LICENSE.md)。

**建议购买正版**：Paul Glasserman, *Monte Carlo Methods in Financial Engineering*, Springer, 2003；中译本《金融工程中的蒙特卡罗方法》。本 notebook 未复制原书任何文字、图表或表格。
